In [1]:
# import dependencies
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

Sensitivity analysis indicates that results are broadly robust to weighting choices, and are more sensitive to the definition of flood risk than to the selection of vulnerability variables. Models using expected annual loss (EAL) produce more stable targeting than those based on simple exposure. Expanding the set of vulnerability indicators adds limited additional signal, suggesting that a parsimonious specification captures most of the relevant variation (Occam's Razor). Based on these results, the EAL-based core model was selected as the final specification.

### Sensitivity Analysis Methodology

- Systematically stress-tested all major index models using two approaches:
    - LOVO (Leave-One-Variable-Out): Dropped each risk and vulnerability variable one at a time to assess the impact on the need index and the set of most underfunded towns.
    - Weight Sensitivity: Varied the weights assigned to risk and vulnerability (e.g., 50/50, 70/30, 30/70) to test the robustness of the index and underfunded town rankings.
- Applied these diagnostics to a range of model specifications, including core, EAL-based, combined, expanded, structural vulnerability, EAL per capita, and FEMA NRI models.
- Compared the stability of the need index (Spearman correlation) and the overlap in the bottom 10 underfunded towns across all tests.

### Key Results

Clear Top Models:
- The core_EAL_model is the most robust and defensible: dropping the risk variable has a moderate effect, vulnerability variables behave as expected, and the model is stable under different weightings.
- The core_model is a strong, interpretable alternative, but is slightly less stable than the EAL version.
- The eal_per_capita_model is more sensitive to risk variable removal, but highlights the influence of population normalization—useful for narrative contrast.

Middle Tier Models:
- The expanded_model shows that adding more vulnerability variables does not materially change results; elderly is especially redundant.
- The combined model appears robust, but in reality, the signal is diffuse; no single variable drives results, making interpretation harder.

General Insights:
- Risk variables drive the index: dropping them causes the largest drop in stability and changes the set of underfunded towns. **The choice of risk variable is the most important modeling decision**—expected loss (EAL) outperforms simple exposure.
- Vulnerability variables matter, but their individual influence is smaller and more stable.
- The need index and underfunded town rankings are robust to reasonable changes in risk/vulnerability weighting.
- More variables do not necessarily make a better model. Parsimony and interpretability win.
- The best models are those that balance strong, interpretable risk signals with a small, meaningful set of vulnerability indicators.
- **The most robust and policy-relevant models are those that combine expected loss (EAL) with core social vulnerability indicators.**

In [2]:
# open csv file
df = pd.read_csv("../data/cleaned/town_level_merged_for_eda.csv")

In [3]:
# data cleaning and preprocessing

# cap extreme outliers in poverty variable (winsorize)
df["pct_below_poverty"] = df["pct_below_poverty"].clip(
    upper=df["pct_below_poverty"].quantile(0.99)
)

# fill funding and claims NaNs with 0
cols_to_fill = [
    "funding_per_capita",
    "log_funding_per_capita",
    "funding_per_occupied_unit",
    "log_funding_per_occupied_unit",
    "claims_paid_per_capita",
    "current_insurance_penetration",
]
df[cols_to_fill] = df[cols_to_fill].fillna(0)

# drop towns with zero population
df = df[df["total_population"] > 0]

# fill remaining NaNs in median_income and median_year_house_built with median values (testing - I may not even use them in the final model)
df["median_income"] = df["median_income"].fillna(df["median_income"].median())
df["median_year_house_built"] = df["median_year_house_built"].fillna(
    df["median_year_house_built"].median()
)

In [4]:
# functions for preprocessing and building indices


def preprocess(df, log_transform=None):
    """
    Preprocess the input DataFrame for modeling.

    - Optionally applies log1p transformation to specified columns.
    - Returns a copy of the DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        log_transform (list or None): List of column names to log-transform (log1p), if provided.

    Returns:
        pd.DataFrame: Preprocessed DataFrame.
    """
    df = df.copy()

    # replace inf values with NaN to avoid issues in modeling
    # df = df.replace([np.inf, -np.inf], np.nan)

    # optional log transformation
    if log_transform:
        for col in log_transform:
            # safeguard against accidentally double-log-transforming variables
            if not col.startswith("log_"):
                df[col] = np.log1p(df[col])

    return df


def z_normalize(df, cols):
    """
    Apply z-score normalization to specified columns in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        cols (list): List of column names to normalize.

    Returns:
        pd.DataFrame: DataFrame with specified columns z-normalized.
    """
    df = df.copy()
    scaler = StandardScaler()
    df[cols] = scaler.fit_transform(df[cols])
    return df


def rank_normalize(df, cols):
    """
    Apply rank-based normalization (percentile ranks) to specified columns in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.
        cols (list): List of column names to normalize.

    Returns:
        pd.DataFrame: DataFrame with specified columns rank-normalized.
    """
    df = df.copy()
    for col in cols:
        df[col] = df[col].rank(pct=True)
    return df


def build_index(df, risk_vars, vuln_vars, funding_var, method="z", log_funding=False):
    """
    Build risk, vulnerability, and need indices, and compute funding gap.

    Args:
        df (pd.DataFrame): Input DataFrame.
        risk_vars (list): List of risk variable column names.
        vuln_vars (list): List of vulnerability variable column names.
        funding_var (str): Column name for funding variable.
        method (str): Normalization method ("z" or "rank").
        log_funding (bool): Whether to log-transform the funding variable.

    Returns:
        pd.DataFrame: DataFrame with new index and gap columns.
    """
    df = df.copy()

    all_vars = risk_vars + vuln_vars

    # normalize risk and vulnerability variables using specified method
    if all_vars:
        df_norm = (
            z_normalize(df, all_vars) if method == "z" else rank_normalize(df, all_vars)
        )
    else:
        df_norm = df.copy()

    # calculate risk and vulnerability indices as the mean of their respective (z-scored) variables
    df["risk_index"] = df_norm[risk_vars].mean(axis=1) if risk_vars else 0
    df["vuln_index"] = df_norm[vuln_vars].mean(axis=1) if vuln_vars else 0

    # combined need index (normalized 0–1-ish if rank, centered if z)
    df["need_index"] = (df["risk_index"] + df["vuln_index"]) / 2

    # # prepare funding series (optional log to reduce skew)
    # # safeguard - I've already log-transformed funding and filled NaNs in preprocessing
    funding = df[funding_var].fillna(0).copy()
    if log_funding:
        funding = np.log1p(funding)

    # scale funding to match chosen normalization method
    if method == "rank":
        # percentile ranks in ~[0,1] to match rank-based indices
        df["funding_scaled"] = funding.rank(pct=True)
    else:
        # z-score to match z-normalized indices (centered around 0)
        df["funding_scaled"] = (
            StandardScaler().fit_transform(funding.values.reshape(-1, 1)).ravel()
        )

    # gap: positive => overfunded relative to need; negative => underfunded
    df["gap_index"] = df["funding_scaled"] - df["need_index"]

    return df

In [5]:
def build_index_weighted(
    df, risk_vars, vuln_vars, funding_var, method="rank", w_risk=0.5, w_vuln=0.5
):
    """
    Build a weighted index combining risk and vulnerability variables, and calculate the gap between funding and need.

    Args:
        df (pd.DataFrame): DataFrame containing the data.
        risk_vars (list): List of risk variable names.
        vuln_vars (list): List of vulnerability variable names.
        funding_var (str): Name of the funding variable.
        method (str): Normalization method, either "rank" or "z".
        w_risk (float): Weight for the risk index.
        w_vuln (float): Weight for the vulnerability index.

    Returns:
        pd.DataFrame: DataFrame with added columns for risk_index, vuln_index, need_index, funding_scaled, and gap_index.
    """

    df = df.copy()
    all_vars = risk_vars + vuln_vars

    # normalize risk and vulnerability variables using specified method
    df_norm = (
        z_normalize(df, all_vars) if method == "z" else rank_normalize(df, all_vars)
    )

    # calculate risk and vulnerability indices as the mean of their respective (z-scored or rank-normalized) variables
    df["risk_index"] = df_norm[risk_vars].mean(axis=1) if risk_vars else 0
    df["vuln_index"] = df_norm[vuln_vars].mean(axis=1) if vuln_vars else 0

    # combined need index as weighted average of risk and vulnerability indices
    df["need_index"] = w_risk * df["risk_index"] + w_vuln * df["vuln_index"]

    # prepare funding series (fill NaNs with 0)
    funding = df[funding_var].fillna(0)

    # scale funding to match chosen normalization method
    if method == "rank":
        # percentile ranks in ~[0,1] to match rank-based indices
        df["funding_scaled"] = funding.rank(pct=True)
    else:
        # z-score to match z-normalized indices (centered around 0)
        df["funding_scaled"] = (
            StandardScaler().fit_transform(funding.values.reshape(-1, 1)).ravel()
        )

    # gap: positive => overfunded relative to need; negative => underfunded
    df["gap_index"] = df["funding_scaled"] - df["need_index"]

    return df

In [6]:
def run_lovo(df, risk_vars, vuln_vars, funding_var, method="rank", top_n=10):
    """
    Run leave-one-variable-out (LOVO) sensitivity analysis on the index construction.
    For each risk and vulnerability variable, drop it from the index construction and see how much the need index and top underfunded towns change.

    Args:
        df (pd.DataFrame): Input DataFrame.
        risk_vars (list): List of risk variable column names.
        vuln_vars (list): List of vulnerability variable column names.
        funding_var (str): Column name for funding variable.
        method (str): Normalization method ("z" or "rank").
        top_n (int): Number of top underfunded towns to consider for overlap analysis.

    Returns:
        pd.DataFrame: DataFrame summarizing the impact of dropping each variable on the need index and top underfunded towns.
    """
    # build the full index with all variables to get the baseline need index and top underfunded towns
    base = build_index(df, risk_vars, vuln_vars, funding_var, method=method)
    base_need = base["need_index"]
    base_bottom = set(base.nsmallest(top_n, "gap_index")["GEOID"])

    # store results for each variable dropped
    results = []

    all_vars = risk_vars + vuln_vars

    # iterate through each variable, drop it from the index construction, and compare the new need index and top underfunded towns to the baseline
    for var in all_vars:
        # create new variable lists with the current variable dropped
        r_vars = [v for v in risk_vars if v != var]
        v_vars = [v for v in vuln_vars if v != var]

        # build new index with the variable dropped
        df_drop = build_index(df, r_vars, v_vars, funding_var, method=method)

        # calculate Spearman correlation between the new need index and the baseline need index to see how much it changes
        need_corr = spearmanr(base_need, df_drop["need_index"])[0]

        # calculate overlap in top underfunded towns between the new index and the baseline
        bottom = set(df_drop.nsmallest(top_n, "gap_index")["GEOID"])
        overlap = len(base_bottom & bottom) / top_n

        # store results
        results.append(
            {"dropped_var": var, "need_spearman": need_corr, "bottom_overlap": overlap}
        )

    return pd.DataFrame(results).sort_values("bottom_overlap")

In [7]:
# define different weight combinations to test in the weighted index sensitivity analysis
weights = [(0.5, 0.5), (0.7, 0.3), (0.3, 0.7)]


def weight_sensitivity(df, risk_vars, vuln_vars, funding_var, top_n=10):
    """
    Test sensitivity of need index and funding gap to different weightings of risk vs vulnerability.

    Args:
        df (pd.DataFrame): Input DataFrame.
        risk_vars (list): List of risk variable column names.
        vuln_vars (list): List of vulnerability variable column names.
        funding_var (str): Column name for funding variable.
        top_n (int): Number of top underfunded towns to consider for overlap analysis.

    Returns:
        pd.DataFrame: DataFrame summarizing the impact of different weightings on the need index and top underfunded towns.
    """
    base = build_index_weighted(
        df, risk_vars, vuln_vars, funding_var, w_risk=0.5, w_vuln=0.5
    )
    base_bottom = set(base.nsmallest(top_n, "gap_index")["GEOID"])

    rows = []

    for wr, wv in weights:
        dfi = build_index_weighted(
            df, risk_vars, vuln_vars, funding_var, w_risk=wr, w_vuln=wv
        )

        corr = spearmanr(base["need_index"], dfi["need_index"])[0]
        bottom = set(dfi.nsmallest(top_n, "gap_index")["GEOID"])
        overlap = len(base_bottom & bottom) / top_n

        rows.append(
            {"w_risk": wr, "w_vuln": wv, "need_corr": corr, "bottom_overlap": overlap}
        )

    return pd.DataFrame(rows)

In [8]:
# select model specifications of interest
model_specs = {
    # single risk variable with core vulnerability variables
    "core_model": {
        "risk": ["pct_river_corridor"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    "core_EAL_model": {
        "risk": ["IFLD_EALT_weighted"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    # all three risk variables together with core vulnerability variables
    "combined": {
        "risk": ["pct_river_corridor", "pct_high_risk_NFHL", "IFLD_EALT_weighted"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    # expanded model with additional vulnerability variables that are commonly used in social vulnerability indices
    "expanded_model": {
        "risk": ["pct_river_corridor", "IFLD_EALT_weighted"],
        "vuln": [
            "pct_below_poverty",
            "percent_elderly",
            "pct_no_vehicle",
            "pct_renter_occupied",
            "pct_mobile_home",
        ],
    },
    # substitute structural vulnerability indicators for socioeconomic ones (physical/demographic fragility)
    "structural_vuln_model": {
        "risk": ["pct_river_corridor"],
        "vuln": [
            "percent_with_disability",
            "pct_mobile_home",
            "median_year_house_built",
        ],
    },
    # EAL decomposed into per capita and asset-normalized versions, to see if they provide more stable or stronger signals than the raw EAL variable
    "eal_per_capita_model": {
        "risk": ["EAL_per_capita"],
        "vuln": ["pct_below_poverty", "percent_elderly", "pct_no_vehicle"],
    },
    # FEMA's own risk and social vulnerability scores, for benchmarking
    "fema_combined_model": {
        "risk": ["RISK_SCORE_avg"],
        "vuln": ["SOVI_SCORE_avg"],
    },
}

In [9]:
# run LOVO and weight sensitivity analyses for each model specification and display results
for name, spec in model_specs.items():
    # extract variable lists for this specification
    risk_vars = spec.get("risk", [])
    vuln_vars = spec.get("vuln", [])
    funding_var = "log_funding_per_capita"

    # run analyses and display results
    print(f"=== {name} ===")
    print("LOVO:")
    display(run_lovo(df, risk_vars, vuln_vars, funding_var, method="rank", top_n=10))
    print("Weight Sensitivity:")
    display(weight_sensitivity(df, risk_vars, vuln_vars, funding_var, top_n=10))

=== core_model ===
LOVO:


,dropped_var,need_spearman,bottom_overlap
0,pct_river_corridor,0.610170,0.3
1,pct_below_poverty,0.962939,0.6
3,pct_no_vehicle,0.956106,0.6
2,percent_elderly,0.958335,0.9


Weight Sensitivity:


,w_risk,w_vuln,need_corr,bottom_overlap
0,0.5,0.5,1.000000,1.0
1,0.7,0.3,0.963124,0.8
2,0.3,0.7,0.928332,0.8


=== core_EAL_model ===
LOVO:


...

,w_risk,w_vuln,need_corr,bottom_overlap
0,0.5,0.5,1.000000,1.0
1,0.7,0.3,0.957955,0.8
2,0.3,0.7,0.918150,0.8


=== fema_combined_model ===
LOVO:


,dropped_var,need_spearman,bottom_overlap
0,RISK_SCORE_avg,0.806662,0.2
1,SOVI_SCORE_avg,0.807861,0.6


Weight Sensitivity:


,w_risk,w_vuln,need_corr,bottom_overlap
0,0.5,0.5,1.000000,1.0
1,0.7,0.3,0.952661,0.7
2,0.3,0.7,0.951211,0.9
